# (노트) 순환신경망

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [빅데이터분석]

### Import 

In [9]:
from fastai.text.all import *
import torch 
import numpy as np

### 실제활용예시

In [10]:
path = untar_data(URLs.IMDB)

In [11]:
files = get_text_files(path)
files

(#100002) [Path('/home/cgb2/.fastai/data/imdb/train/neg/5300_4.txt'),Path('/home/cgb2/.fastai/data/imdb/train/neg/6638_4.txt'),Path('/home/cgb2/.fastai/data/imdb/train/neg/6860_2.txt'),Path('/home/cgb2/.fastai/data/imdb/train/neg/5802_3.txt'),Path('/home/cgb2/.fastai/data/imdb/train/neg/6527_1.txt'),Path('/home/cgb2/.fastai/data/imdb/train/neg/7401_1.txt'),Path('/home/cgb2/.fastai/data/imdb/train/neg/6544_1.txt'),Path('/home/cgb2/.fastai/data/imdb/train/neg/1162_1.txt'),Path('/home/cgb2/.fastai/data/imdb/train/neg/7267_1.txt'),Path('/home/cgb2/.fastai/data/imdb/train/neg/7674_1.txt')...]

- 경로안에 train, test, unsup이라는 폴더가 있는데, 이 폴더들내의 모든 텍스트파일을 가져오라. 

In [12]:
dls=DataBlock(
    blocks=TextBlock.from_folder(path,is_lm=True),
    get_items=get_text_files,splitter=RandomSplitter(0.1)
).dataloaders(source=path, bs=128, seq_len=80)

- `is_lm`: 문장생성을 위해서는 is_lm=True로 설정해야함.   
- splitter: train / val 으로 나눔 

In [13]:
dls.show_batch()

,text,text_
0,"xxbos xxmaj this film tackles the subjects of loss , personal struggle and transformation in such a smart , artful , sensitive , and visually stunning way that i was completely transported . xxmaj it is a rare gem of a film in the way it honors beauty and women . xxmaj you 'll have to see for yourself . xxmaj dreya xxmaj weber ( jane ) masterfully portrays the subtleties of a remarkable if not somewhat broken personality ,","xxmaj this film tackles the subjects of loss , personal struggle and transformation in such a smart , artful , sensitive , and visually stunning way that i was completely transported . xxmaj it is a rare gem of a film in the way it honors beauty and women . xxmaj you 'll have to see for yourself . xxmaj dreya xxmaj weber ( jane ) masterfully portrays the subtleties of a remarkable if not somewhat broken personality , in"
1,"xxmaj we watch them to see xxmaj jackie xxmaj chan perform eye - popping stunts and generally punch and kick a whole lot of people who are less awesome than he is . \n\n xxmaj and on that level , ' police xxmaj story ' does n't disappoint at all . xxmaj there is an awesome showdown in a shopping mall at the end that culminates in xxmaj jackie sliding down a giant , high - voltage chandelier . xxmaj","we watch them to see xxmaj jackie xxmaj chan perform eye - popping stunts and generally punch and kick a whole lot of people who are less awesome than he is . \n\n xxmaj and on that level , ' police xxmaj story ' does n't disappoint at all . xxmaj there is an awesome showdown in a shopping mall at the end that culminates in xxmaj jackie sliding down a giant , high - voltage chandelier . xxmaj seriously"
2,"on the collagen before they take over her face , they 're starting to look like she was beaten up . \n\n the xxup sfx would have embarrassed the xxmaj red xxmaj dwarf crew at times they were so bad . xxmaj for an example of how bad it gets check out the scene on the ship at warp where the view out the window is n't even xxup cgi but some space scene on a long roller rolled past","the collagen before they take over her face , they 're starting to look like she was beaten up . \n\n the xxup sfx would have embarrassed the xxmaj red xxmaj dwarf crew at times they were so bad . xxmaj for an example of how bad it gets check out the scene on the ship at warp where the view out the window is n't even xxup cgi but some space scene on a long roller rolled past the"
3,and history before putting ' xxunk ' anywhere near the center of the issue . i could write a lot more but i guess it 's just not worth it . xxmaj my recommendation to those willing to watch this : do n't bother . xxmaj better buy a ticket to xxmaj bangkok and see the amazing city and amazing country for yourself . xxmaj and there i no need to be afraid to get to the places where those,history before putting ' xxunk ' anywhere near the center of the issue . i could write a lot more but i guess it 's just not worth it . xxmaj my recommendation to those willing to watch this : do n't bother . xxmaj better buy a ticket to xxmaj bangkok and see the amazing city and amazing country for yourself . xxmaj and there i no need to be afraid to get to the places where those girls
4,"wo n't give out any details , but things happen that xxup i , for one , have no desire to see . i like to give all movies the benefit of the doubt , and i really wanted to like this one . xxmaj it just did n't work out . i give it a 3 out of 10 , mainly for the acting . xxbos i saw the xxmaj german version of the movie in xxmaj german television","n't give out any details , but things happen that xxup i , for one , have no desire to see . i like to give all movies the benefit of the doubt , and i really wanted to like this one . xxmaj it just did n't work out . i give it a 3 out of 10 , mainly for the acting . xxbos i saw the xxmaj german version of the movie in xxmaj german television and"
5,"paradise when these movies can only be found if you buy them .

- xxbox: 새로운 텍스트의 시작 
- xxmaj: 다음단어가 대문자로 시작함을 의미 (모든단어는 기본적으로 소문자라고 생각함) 
- .. 

In [14]:
lrnr = language_model_learner(dls,AWD_LSTM,metrics=accuracy).to_fp16()

EOFError: Compressed file ended before the end-of-stream marker was reached

- to_fp16(): 그레디언트를 대충계산한다. 

In [7]:
lrnr.fit_one_cycle(5)

epoch,train_loss,valid_loss,accuracy,time
0,4.462815,4.136233,0.283587,07:26
1,4.335924,4.033761,0.290330,07:23
2,4.293494,3.999372,0.292693,07:22
3,4.274117,3.986934,0.293585,07:21
4,4.258859,3.984887,0.293765,07:36


In [8]:
TEXT = 'I liked this movie because' 
N_WORDS= 40 
N_SENTENCES = 5
preds = [lrnr.predict(TEXT,N_WORDS,temperature=0.75) for _ in range(N_SENTENCES)]
print("\n".join(preds))

i liked this movie because i had already heard of the Academy Award for this movie and i loved it . It was truly one of the best movies i have ever seen . i was pretty happy with it . i
i liked this movie because i wanted to be a fan of B 6.5 movies in the early 80 's and lived for a long time . It was so interesting that i was hoping for it to make it it 's "
i liked this movie because it was a fun movie . i thought it was a cheap , gritty movie about a man who has been around for years , his father is a film maker , i believe he picked out the Dutch
i liked this movie because they were watching this film on TV , but this was one of the worst movies ever made . So , i tried not to throw in this movie . The plot is simple , and this
i liked this movie because i liked Jaws when there was a plot and it was a great movie . The only flaw in this movie is that there is nothing going on that he does not even make it … a very


- 말이 안되는것 같은데 그럴듯해보이긴함 

### 문제의설계 

In [ ]:
text= 'h e l l o '*100
text

In [ ]:
tokens= text.split(' ')[:-1]
tokens[:10]

`-` 바로직전문자로 다음문자를 맞춰보자. 
- hello 니까, h $\to$ e, e $\to$ l, l $\to$ l/o (?), l $\to$ o, o $\to$ h ... 
- l 다음에 올 문자가 조금 애매하다. 

`-` 마치 아래의 표에서 $X \to y$인 맵핑을 알아차려 $X$를 보고 $y$를 예측하듯이 

|X|y|
|:-:|:-:|
|1|2|
|2|4|
|3|6|
|1|2|
|2|4|
|3|6|
|...|...|

우리는 이제 아래를 예측하는 규칙을 알아차리는게 목표이다. 

|X|y|
|:-:|:-:|
|h|e|
|e|l|
|l|l/o|
|o|h|
|h|e|
|...|...|

### Embedding 

`-` X,y를 설정하자.  

In [ ]:
X= tokens[:(len(tokens)-1)]
y= tokens[1:len(tokens)]

In [ ]:
X[0],y[0]

In [ ]:
X[1],y[1]

`-` 다 좋은데 학습가능한 형태로 만들어야 하므로 문자를 숫자로 바꾸자. 

In [ ]:
dic = {'h': 0, 'e': 1, 'l': 2, 'o': 3}
dic

In [ ]:
dic['h'],dic['e'],dic['l'],dic['o'] 

In [ ]:
nums = [dic[i] for i in tokens]

In [ ]:
tokens[:10],nums[:10]

`-` (맵핑방식1) 아래와 같이 문자와 숫자를 mapping 하였다.

|문자(tokens)|숫자(nums)|
|:-:|:-:|
|'h'|0|
|'e'|1|
|'l'|2|
|'l'|2|
|'o'|3|
|'h'|0|
|'e'|1|
|'l'|2|
|'l'|2|
|'o'|3|
|'h'|0|
|'e'|1|
|'l'|2|
|'l'|2|
|'o'|3|
|...|...|

`-` (맵핑방식2) 사실 아래와 같이 맵핑하는 것이 더 좋다. 위의방식대로하면 의미가 `e=1`, `l=2`가 되는데 그렇다고해서 `l이 e보다 2배가 강한 입력`을 의미하는것은 아니므로.. 

|문자(tokens)|숫자(nums)|
|:-:|:-:|
|'h'|1,0,0,0|
|'e'|0,1,0,0|
|'l'|0,0,1,0|
|'l'|0,0,1,0|
|'o'|0,0,0,1|
|'h'|1,0,0,0|
|'e'|0,1,0,0|
|'l'|0,0,1,0|
|'l'|0,0,1,0|
|'o'|0,0,0,1|
|'h'|1,0,0,0|
|'e'|0,1,0,0|
|'l'|0,0,1,0|
|'l'|0,0,1,0|
|'o'|0,0,0,1|
|...|...|

`-` 맵핑방식2로 처리하고 싶은데, 데이터 전처리 하기가 너무 하기 어려울것 같다. 
- 그런데 이러한것은 빈번하게 일어나는 상황아닌가?.. 
- 누군가 구해놓지 않았을까?.. 
- torch.nn.Embedding 

`-` 맵핑방식1의 구현 

In [ ]:
_l1 = torch.nn.Linear(in_features=1,out_features=20,bias=False)

In [ ]:
_x = torch.tensor([[0.0],[1.0],[2.0],[2.0],[3.0],[0.0],[1.0],[2.0],[2.0],[3.0]])
_l1(_x)

- 입력: (10,1) 
- 출력: (10,20) 

`-` 맵핑방식2의 구현 

In [ ]:
e1=torch.nn.Embedding(num_embeddings=4,embedding_dim=20) 

In [8]:
_x = torch.tensor([0,1,2,3,0,1,2,3])
e1(_x)

NameError: name 'e1' is not defined

- 입력차원: (10,1) 
- 출력차원: (10,20)

`-` 결과가 똑같다? 하지만 내부 수행과정은 약간 다르다. 

In [9]:
list(_l1.parameters())

NameError: name '_l1' is not defined

- 맵핑방식1의 경우 $1\times 20$의 차원의 ${\bf W}$가 존재함. 따라서 
    - ${\bf X}$: (10,1) 
    - ${\bf W}$: (1,20) 
    - ${\bf X}{\bf W}$: (10,20)

In [ ]:
list(e1.parameters())

- 맵핑방식2의 경우 $4\times 20$의 차원의 ${\bf W}$가 존재함. 따라서 
    - ${\bf X}$: (10,1) 
    - $\tilde{{\bf X}}$: (10,4) 
    - ${\bf W}$: (4,20) 
    - $\tilde{{\bf X}}{\bf W}$: (10,20)

`-` 결국 우리가 맵핑방식2처럼 구현하고 싶다고 하더라도, 입력은 아래와 같이 넣어도 무방하다. 이후에는 알아서 파이토치가 해결해준다. 

In [ ]:
_x

### 네트워크 구축 

`-` 이제 숫자화된 자료를 이용하여 다시 X,y를 선언하자. 

In [ ]:
X = torch.tensor(nums[:499])
y = torch.tensor(nums[1:])

In [ ]:
X[0],y[0]

In [ ]:
X[1],y[1]

`-` 간단한 네트워크를 통하여 학습을 시도하자. 

In [ ]:
e1=torch.nn.Embedding(num_embeddings=4,embedding_dim=20) 
l1=torch.nn.Linear(in_features=20,out_features=20)
a1=torch.nn.ReLU()
l2=torch.nn.Linear(in_features=20,out_features=4)
a2=torch.nn.Softmax()

In [ ]:
X.shape,e1(X).shape

In [ ]:
e1(X).shape,a1(l1(e1(X))).shape

In [ ]:
a1(l1(e1(X))).shape,l2(a1(l1(e1(X)))).shape

In [ ]:
l2(a1(l1(e1(X))))[0]

In [ ]:
a2(l2(a1(l1(e1(X)))))[0]

- $X$의 차원이 명시되지 않아서 대충 내가 알아서 계산했다는 워닝

`-` 경고가 찝찝하여 손계산해봄 $\to$ 알아서 잘 계산했음 

In [ ]:
np.exp(0.1195)/(np.exp(0.1195)+np.exp(-0.3600)+np.exp(-0.1282)+np.exp(-0.5230))

`-` 순전파차원변화 요약 

```
torch.Size([499]) ## X with levels=4 
torch.Size([499, 20]) ## 임베딩 이후
torch.Size([499, 20]) ## 선형변환1 이후
torch.Size([499, 20]) ## 렐루 이후
torch.Size([499, 4]) ## 선형변환2 이후 
torch.Size([499, 4]) ## 소프트맥스 이후 = yhat 
```

In [ ]:
net = torch.nn.Sequential(
    torch.nn.Embedding(num_embeddings=4,embedding_dim=20),
    torch.nn.Linear(in_features=20,out_features=20),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=20,out_features=4))
    #torch.nn.Softmax()

In [ ]:
net(X)

- 순전파 

`-` 손실함수, 옵티마이저 

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(net.parameters())

In [ ]:
for i in range(1000):
    ## 1 
    yhat = net(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward()
    ## 4 
    optimizer.step()
    optimizer.zero_grad()

In [ ]:
dic

In [ ]:
X[:7] 

In [ ]:
net(X)[:7]

In [ ]:
y[:7]

- 학습이 잘 되었다. 

### net의 개선 

`-` 단어수가 바뀔때마다 아래를 새로 정의해야 하나?
```python
net = torch.nn.Sequential(
    torch.nn.Embedding(num_embeddings=4,embedding_dim=20),
    torch.nn.Linear(in_features=20,out_features=20),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=20,out_features=4))
    #torch.nn.Softmax()
```

`-` 네트워크를 찍어내는 뭔가가 있음 좋지 않을까? 제가 만들어볼게요!

In [ ]:
class BDA(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        u= self.linear1(self.embedding(X))
        v= self.relu(u)
        return self.linear2(v)

- 그냥 이렇게 간단해보이는 코드로 가능하다고? $\to$ 필요한 다른기능들은 Module에서 상속받았으므로 가능하다!

In [ ]:
net2 = BDA(4) # 4는 dic의 size

In [ ]:
net

In [ ]:
net2

- 오.. 

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss() 
optimizer2 = torch.optim.Adam(net2.parameters())

In [ ]:
for i in range(1000):
    ## 1 
    yhat = net2(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward()
    ## 4 
    optimizer2.step()
    optimizer2.zero_grad()

In [ ]:
net2(X)

In [ ]:
net(X)

`-` net2도 잘 학습 되었다.

### 이전2단계

In [ ]:
X = torch.tensor([nums[:498],nums[1:499]]).T
y = torch.tensor(nums[2:])

In [ ]:
X[0],y[0] # he -> l 

In [ ]:
X[1],y[1] # el -> l 

In [ ]:
X[2],y[2] # ll -> o

`-` 아키텍처를 대충 스케치해보자. 

In [ ]:
_e1 = torch.nn.Embedding(num_embeddings=4,embedding_dim=20)

In [ ]:
X.shape, _e1(X).shape

`-` 이전의 아키텍처는 아래와 같았음. 

```
torch.Size([499]) ## X with levels=4 
torch.Size([499, 20]) ## 임베딩 이후
torch.Size([499, 20]) ## 선형변환1 이후
torch.Size([499, 20]) ## 렐루 이후
torch.Size([499, 4]) ## 선형변환2 이후 
torch.Size([499, 4]) ## 소프트맥스 이후 = yhat 
```

`-` 벌써부터 꼬이는데?? 마지막 차원은 어쩌지? 합치나? 
- 합치는게 나쁜건 아닐지 몰라도 다른 대안이 있음 $\to$ 순환신경망의 발견 

`-` 순환신경망의 설계 

In [ ]:
class BDA2(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        x1=X[:,0] # X의 첫번째 칼럼, y보다 2시점이전
        x2=X[:,1] # X의 두번째 칼럼, y보다 1시점이전 
        v=self.relu(self.linear1(self.embedding(x1))) # v: (498,20) 
        v2=self.relu(self.linear1(v+self.embedding(x2))) # 
        return self.linear2(v2)

`-` 결국 최종출력인 v2는 v와 x2이 담긴 함수이다. 그런데 v는 x1이 담긴 함수이다. 따라서 v2는 x2가 담겨있는 동시에 x1도 약하게 담겨있다 볼 수 있음. 

In [ ]:
net3=BDA2(4)
net3

In [ ]:
net2

- 구조의 차이는 없지만 순전파 계산방식이 다르다! (그렇다면 역전파 계산방식도 다르겠지?..) 

`-` 다시 학습해보자. 

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss() # 사실손실함수는 계속 재활용해도 괜찮은데.. 어차피 같은거임 
optimizer3 = torch.optim.Adam(net3.parameters())

In [ ]:
for i in range(1000):
    ## 1 
    yhat = net3(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward() 
    ## 4 
    optimizer3.step()
    optimizer3.zero_grad()

In [ ]:
X[:5]

In [ ]:
net3(X)[:5]

- he $\to$ l 
- el $\to$ l 
- ll $\to$ o 
- lo $\to$ h
- oh $\to$ e 

`-` 학습이 잘되었다. 

`-` 네트워크를 만드는 클래스를 조금 정리하자. 

```python 
### 원래 이런클래스였음 
class BDA2(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        x1=X[:,0] # X의 첫번째 칼럼, y보다 2시점이전
        x2=X[:,1] # X의 두번째 칼럼, y보다 1시점이전 
        v=self.relu(self.linear1(self.embedding(x1))) # v: (498,20) 
        v2=self.relu(self.linear1(v+self.embedding(x2))) # 
        return self.linear2(v2)

```

In [ ]:
class BDA2(Module): ## 조금 정리한 클래스
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        v=0 
        for i in range(2):        
            v=self.relu(self.linear1(v+self.embedding(X[:,i]))) # 
        return self.linear2(v)

In [ ]:
net3=BDA2(4)
loss_fn = torch.nn.CrossEntropyLoss() # 사실손실함수는 계속 재활용해도 괜찮은데.. 어차피 같은거임 
optimizer3 = torch.optim.Adam(net3.parameters())

In [ ]:
for i in range(1000):
    ## 1 
    yhat = net3(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward() 
    ## 4 
    optimizer3.step()
    optimizer3.zero_grad()

In [ ]:
X[:5]

In [ ]:
net3(X)[:5]

- 동일한 결과가 나옴. 코드정리가 성공적이었음. 

### 발전 